# Build Sentiment Tables

This notebook adapts lexicon-based sentiment analysis to the constitution project by using the Loughran-McDonald dictionary for legal-institutional tone. It builds three reusable project tables:

- `vocab_sent.parquet`
- `bow_sent.parquet`
- `doc_sent.parquet`

It also creates sentiment visualizations tied to metadata in `LIB` and exports the main time-series view as HTML for embedding.

## Inputs and Outputs

**Input files**
- `lib.csv`
- `vocab.csv`
- `bow.parquet`
- `Loughran-McDonald_MasterDictionary_1993-2025.csv`

**Output files**
- `vocab_sent.parquet`
- `bow_sent.parquet`
- `doc_sent.parquet`
- `sentiment_dimensions_by_year_created.html`

The notebook uses the Loughran-McDonald dictionary to attach lexical sentiment dimensions to the project vocabulary, aggregates those dimensions from bag-term counts back to article-level and constitution-level summaries keyed by `country_id`, and then visualizes how those dimensions vary across time, region, and regime metadata.

This workflow is justified because the article-level bags preserve local constitutional tone across more than 33,000 articles, while constitution-level summaries make cross-country comparisons readable and reduce the bias that would come from longer constitutions simply containing more tokens. The tradeoff is that country-level averages can flatten sharp tonal contrasts across articles within the same constitution.


## Set Up

The notebook expects the core project files in the working directory plus a local copy of the Loughran-McDonald dictionary. The loader below checks a few candidate paths and raises a clear error if no matching file is found.

In [1]:
# Maintainer note: this notebook adapts the sentiment-table pattern from `uva-ds-5001-m10-sentiment-analysis-of-novels.ipynb`.
# Keep that source note here in code comments instead of the final-project-facing documentation.
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

LIB_CSV = Path('lib.csv')
VOCAB_CSV = Path('vocab.csv')
BOW_PARQUET = Path('bow.parquet')

VOCAB_SENT_PARQUET = Path('vocab_sent.parquet')
BOW_SENT_PARQUET = Path('bow_sent.parquet')
DOC_SENT_PARQUET = Path('doc_sent.parquet')

LM_CANDIDATES = [
    Path('Loughran-McDonald_MasterDictionary_1993-2025.csv'),
    Path('lexicons/Loughran-McDonald_MasterDictionary_1993-2025.csv'),
    Path('data/lexicons/Loughran-McDonald_MasterDictionary_1993-2025.csv'),
    Path('sentiment/Loughran-McDonald_MasterDictionary_1993-2025.csv'),
]

DEFAULT_TIME_COL = 'year_created'
DEFAULT_SENTIMENT_COL = 'sentiment_mean'


## Helper Functions

In [2]:
def find_existing_path(candidates):
    # Use the first locally available lexicon path so the notebook can run in different workspaces.
    for path in candidates:
        if path.exists():
            return path
    checked = '\n- '.join(str(path) for path in candidates)
    raise FileNotFoundError(
        'Could not find a local Loughran-McDonald sentiment lexicon. Place the CSV in one of these locations:\n'
        f'- {checked}'
    )


def standardize_lm(lm: pd.DataFrame) -> pd.DataFrame:
    # Convert the Loughran-McDonald master dictionary into a compact term-by-feature table.
    lm = lm.copy()
    if 'Word' not in lm.columns:
        raise ValueError('The Loughran-McDonald file must contain a `Word` column.')
    lm['term_str'] = lm['Word'].str.lower()
    keep_cols = [
        'term_str', 'Negative', 'Positive', 'Uncertainty', 'Litigious',
        'Strong_Modal', 'Weak_Modal', 'Constraining', 'Complexity'
    ]
    lm = lm[keep_cols].copy()
    rename_map = {
        'Negative': 'negative',
        'Positive': 'positive',
        'Uncertainty': 'uncertainty',
        'Litigious': 'litigious',
        'Strong_Modal': 'strong_modal',
        'Weak_Modal': 'weak_modal',
        'Constraining': 'constraining',
        'Complexity': 'complexity'
    }
    lm = lm.rename(columns=rename_map)
    # Collapse category counts to binary term membership because the corpus model is count-based.
    for col in rename_map.values():
        lm[col] = (lm[col] > 0).astype(int)
    # Derive a simple net polarity score so positive and negative can be combined in one dimension.
    lm['sentiment'] = lm['positive'] - lm['negative']
    return lm.set_index('term_str').sort_index()


def infer_bag_cols(bow: pd.DataFrame) -> list[str]:
    # Reuse the existing project bagging scheme instead of hard-coding article-level assumptions.
    candidates = ['country_id', 'article_n', 'clause_n']
    bag_cols = [col for col in candidates if col in bow.columns]
    if not bag_cols:
        raise ValueError('Could not infer bag columns from the BOW table.')
    return bag_cols


def prepare_weighted_sentiment(bow_sent: pd.DataFrame, sentiment_cols: list[str]) -> pd.DataFrame:
    # Multiply each lexicon flag by the bag-term count so later aggregation is token-weighted.
    bow_sent = bow_sent.copy()
    for col in sentiment_cols:
        bow_sent[f'{col}_w'] = bow_sent['n'] * bow_sent[col]
    return bow_sent


## Load Inputs

The `BOW` table already matches the project bagging scheme, so this notebook maps sentiment values onto those bag-term counts instead of rebuilding document strings. It also uses the `VOCAB.stop` flag so English stopwords and constitution-specific legal boilerplate from `build_vocab.ipynb` are excluded before sentiment is aggregated.

In [3]:
# Load the sentiment lexicon first so the active dimensions are known before table building.
LM_PATH = find_existing_path(LM_CANDIDATES)
LM = standardize_lm(pd.read_csv(LM_PATH))

LIB = pd.read_csv(LIB_CSV).set_index('country_id').sort_index()
VOCAB = pd.read_csv(VOCAB_CSV).set_index('term_str').sort_index()
# Respect the project stopword and legal-boilerplate filtering before matching lexicon terms.
VOCAB_ACTIVE = VOCAB[VOCAB['stop'] == 0].copy()
BOW = pd.read_parquet(BOW_PARQUET)

BAG_COLS = infer_bag_cols(BOW)
SENTIMENT_COLS = [col for col in LM.columns if pd.api.types.is_numeric_dtype(LM[col])]

print(f'Sentiment lexicon: {LM_PATH.resolve()}')
print(f'Active non-stop vocabulary size: {VOCAB_ACTIVE.shape[0]:,}')
print(f'BOW bag columns: {BAG_COLS}')
print(f'Sentiment columns: {SENTIMENT_COLS}')

Sentiment lexicon: C:\Users\garre\school\spring_2026\ds_5001\Loughran-McDonald_MasterDictionary_1993-2025.csv
Active non-stop vocabulary size: 24,681
BOW bag columns: ['country_id', 'article_n']
Sentiment columns: ['negative', 'positive', 'uncertainty', 'litigious', 'strong_modal', 'weak_modal', 'constraining', 'complexity', 'sentiment']


In [4]:
display(LM.head())
display(BOW.head())

,negative,positive,uncertainty,litigious,strong_modal,weak_modal,constraining,complexity,sentiment
term_str,,,,,,,,,
aardvark,0,0,0,0,0,0,0,0,0
aardvarks,0,0,0,0,0,0,0,0,0
abaci,0,0,0,0,0,0,0,0,0
aback,0,0,0,0,0,0,0,0,0
abacus,0,0,0,0,0,0,0,0,0


,country_id,article_n,term_str,n,tfidf
0,Afghanistan,1,accordance,1,2.950516
1,Afghanistan,1,adhering,1,9.123469
2,Afghanistan,1,admiring,1,10.732907
3,Afghanistan,1,afghanistan,3,22.602701
4,Afghanistan,1,all,3,8.100179


## Build `VOCAB_SENT`

This table keeps the subset of the non-stop project `VOCAB` that appears in the Loughran-McDonald dictionary. It preserves the project vocabulary statistics and adds the dictionary's legal-financial sentiment dimensions.

In [5]:
# Keep only vocabulary terms that survive project filtering and appear in the lexicon.
VOCAB_SENT = VOCAB_ACTIVE.join(LM, how='inner')
VOCAB_SENT['matched_lexicon'] = 1

print(f'VOCAB shape: {VOCAB.shape}')
print(f'VOCAB after stop filtering: {VOCAB_ACTIVE.shape}')
print(f'VOCAB_SENT shape: {VOCAB_SENT.shape}')
print(f'Lexicon coverage of active vocabulary: {VOCAB_SENT.shape[0] / VOCAB_ACTIVE.shape[0]:.2%}')
VOCAB_SENT.head()

VOCAB shape: (24735, 15)
VOCAB after stop filtering: (24681, 15)
VOCAB_SENT shape: (15294, 25)
Lexicon coverage of active vocabulary: 61.97%


,n,p,i,df,idf,dfidf,n_chars,max_pos,max_pos_group,n_pos,...,negative,positive,uncertainty,litigious,strong_modal,weak_modal,constraining,complexity,sentiment,matched_lexicon
term_str,,,,,,,,,,,,,,,,,,,,,
abandon,11,2.864780e-06,18.413144,11,5.007495,55.082440,7,VB,VB,2,...,1,0,0,0,0,0,0,0,-1,1
abandoned,35,9.115210e-06,16.743293,27,3.785102,102.197757,9,VBN,VB,8,...,1,0,0,0,0,0,0,0,-1,1
abandoning,1,2.604346e-07,21.872576,1,7.592457,7.592457,10,VBG,VB,1,...,1,0,0,0,0,0,0,0,-1,1
abandonment,40,1.041738e-05,16.550648,32,3.548063,113.538013,11,NN,NN,4,...,1,0,0,0,0,0,0,0,-1,1
abandons,4,1.041738e-06,19.872576,3,6.592457,19.777371,8,VBZ,VB,1,...,1,0,0,0,0,0,0,0,-1,1


## Build `BOW_SENT`

This table maps the lexicon-backed sentiment values onto the project `BOW`. Each row is still one bag-term pair, but now the row also carries its dictionary values and weighted count versions such as `sentiment_w = n * sentiment`.

In [6]:
# Map lexicon features from the vocabulary table onto the long bag-of-words table.
VOCAB_SENT_JOIN = VOCAB_SENT.reset_index()[['term_str'] + SENTIMENT_COLS]
BOW_SENT = BOW.merge(VOCAB_SENT_JOIN, on='term_str', how='inner')
BOW_SENT = prepare_weighted_sentiment(BOW_SENT, SENTIMENT_COLS)

print(f'BOW shape: {BOW.shape}')
print(f'BOW_SENT shape: {BOW_SENT.shape}')
print(f'Sentiment-bearing BOW rows: {BOW_SENT.shape[0] / BOW.shape[0]:.2%}')
BOW_SENT.head()

BOW shape: (1775569, 5)
BOW_SENT shape: (1307209, 23)
Sentiment-bearing BOW rows: 73.62%


,country_id,article_n,term_str,n,tfidf,negative,positive,uncertainty,litigious,strong_modal,...,sentiment,negative_w,positive_w,uncertainty_w,litigious_w,strong_modal_w,weak_modal_w,constraining_w,complexity_w,sentiment_w
0,Afghanistan,1,accordance,1,2.950516,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Afghanistan,1,adhering,1,9.123469,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Afghanistan,1,admiring,1,10.732907,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Afghanistan,1,all,3,8.100179,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Afghanistan,1,almighty,1,6.597740,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Build `DOC_SENT`

This table aggregates sentiment from `BOW_SENT` back to the bag level. Because the current project `BOW` is article-based, each row in `DOC_SENT` represents one `country_id | article_n` bag with linked metadata from `LIB`.

Those article-level rows are the bridge between the token counts and the later constitution-level summaries. They preserve local variation in tone before any averaging back to one row per constitution for visualization.


In [7]:
# Aggregate token-weighted lexicon hits back to one row per bag.
weighted_cols = [f'{col}_w' for col in SENTIMENT_COLS]

DOC_SENT = (
    BOW_SENT.groupby(BAG_COLS)
    .agg(
        matched_token_count=('n', 'sum'),
        matched_term_count=('term_str', 'nunique'),
        **{col: (col, 'sum') for col in weighted_cols}
    )
    .reset_index()
)

for col in SENTIMENT_COLS:
    # Convert weighted sums into average per-matched-token rates for cross-document comparison.
    DOC_SENT[f'{col}_mean'] = DOC_SENT[f'{col}_w'] / DOC_SENT['matched_token_count']

DOC_SENT = DOC_SENT.merge(LIB.reset_index(), on='country_id', how='left')
DOC_SENT = DOC_SENT.sort_values(BAG_COLS).reset_index(drop=True)

print(f'DOC_SENT shape: {DOC_SENT.shape}')
DOC_SENT.head()

DOC_SENT shape: (33725, 40)


,country_id,article_n,matched_token_count,matched_term_count,negative_w,positive_w,uncertainty_w,litigious_w,strong_modal_w,weak_modal_w,...,line_count,article_count,clause_count,token_count,v2x_regime,v2x_freexp_altinf,v2x_rule,v2x_regime_cat,v2x_freexp_altinf_cat,v2x_rule_cat
0,Afghanistan,1,143,119,3,8,1,4,0,0,...,1482,163,277,9937,1.0,0.666,0.122,Electoral autocracy,High,Very low
1,Afghanistan,2,5,5,0,0,0,0,0,0,...,1482,163,277,9937,1.0,0.666,0.122,Electoral autocracy,High,Very low
2,Afghanistan,3,15,14,0,0,0,1,0,0,...,1482,163,277,9937,1.0,0.666,0.122,Electoral autocracy,High,Very low
3,Afghanistan,4,6,6,0,0,0,2,0,0,...,1482,163,277,9937,1.0,0.666,0.122,Electoral autocracy,High,Very low
4,Afghanistan,5,35,30,1,0,0,2,0,0,...,1482,163,277,9937,1.0,0.666,0.122,Electoral autocracy,High,Very low


## Quick Summaries

In [8]:
summary_cols = [
    col for col in [
        'sentiment_mean', 'positive_mean', 'negative_mean', 'uncertainty_mean',
        'litigious_mean', 'strong_modal_mean', 'weak_modal_mean', 'constraining_mean'
    ] if col in DOC_SENT.columns
]
DOC_SENT[summary_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
sentiment_mean,33725.0,-0.023887,0.064450,-1.0,-0.040000,0.000000,0.000000,0.285714
positive_mean,33725.0,0.008449,0.020888,0.0,0.000000,0.000000,0.004202,0.400000
negative_mean,33725.0,0.032336,0.059831,0.0,0.000000,0.004975,0.045455,1.000000
uncertainty_mean,33725.0,0.015085,0.026926,0.0,0.000000,0.000000,0.023256,1.000000
litigious_mean,33725.0,0.078932,0.087950,0.0,0.012346,0.057143,0.111111,1.000000
strong_modal_mean,33725.0,0.002951,0.012288,0.0,0.000000,0.000000,0.000000,0.285714
weak_modal_mean,33725.0,0.013113,0.025073,0.0,0.000000,0.000000,0.019417,1.000000
constraining_mean,33725.0,0.010310,0.023488,0.0,0.000000,0.000000,0.010309,0.400000


In [9]:
# Rank constitutions by their average bag-level net polarity for quick inspection.
country_sentiment = (
    DOC_SENT.groupby('country_id')[DEFAULT_SENTIMENT_COL]
    .mean()
    .sort_values(ascending=False)
    .rename('mean_bag_sentiment')
    .to_frame()
)

display(country_sentiment.head(10))
display(country_sentiment.tail(10))

,mean_bag_sentiment
country_id,
People's Republic of Korea,0.007109
Guinea-Bissau,-0.001043
Socialist Republic of Vietnam,-0.001326
Equatorial Guinea,-0.002432
Laos,-0.002704
Libya,-0.004795
China,-0.006427
Djibouti,-0.007020
Iraq,-0.009203


,mean_bag_sentiment
country_id,
Haiti,-0.038315
Sweden,-0.038352
Mexico,-0.039567
Micronesia,-0.042052
Lebanon,-0.043857
United States of America,-0.044558
Slovakia,-0.047021
Israel,-0.051264
Luxembourg,-0.062826


## Sentiment Visualizations

The charts below use country_id-level mean sentiment so constitutions with many articles do not dominate the metadata comparisons. In this notebook, the headline time plot uses `sentiment_mean`, which is the mean net polarity score derived from Loughran-McDonald as `positive_mean - negative_mean`. The additional time-series subplots break that net score into more specific dimensions so you can see whether changes come from positive language, negative language, uncertainty language, or constraining language.

In [10]:
# Average article-level sentiment up to the constitution level before plotting metadata relationships.
country_mean_cols = [col for col in DOC_SENT.columns if col.endswith('_mean')]
COUNTRY_SENT = (
    DOC_SENT.groupby('country_id')[country_mean_cols]
    .mean()
    .reset_index()
    .merge(LIB.reset_index(), on='country_id', how='left')
)

# Plot mean net polarity over constitution creation years.
sentiment_over_time = (
    COUNTRY_SENT.dropna(subset=[DEFAULT_TIME_COL])
    .groupby(DEFAULT_TIME_COL)[DEFAULT_SENTIMENT_COL]
    .mean()
    .reset_index()
    .sort_values(DEFAULT_TIME_COL)
)

fig_time = px.line(
    sentiment_over_time,
    x=DEFAULT_TIME_COL,
    y=DEFAULT_SENTIMENT_COL,
    markers=True,
    title='Mean Constitution Sentiment by Year Created'
)
fig_time.update_layout(
    xaxis_title='Year Created',
    yaxis_title='Mean Sentiment',
    template='plotly_white'
)
fig_time.show()

# Also show the main Loughran-McDonald dimensions separately so net sentiment is easier to interpret.
time_feature_cols = [
    col for col in ['positive_mean', 'negative_mean', 'uncertainty_mean', 'litigious_mean', 'constraining_mean']
    if col in COUNTRY_SENT.columns
]
time_series_by_feature = {}
for feature_col in time_feature_cols:
    time_series_by_feature[feature_col] = (
        COUNTRY_SENT.dropna(subset=[DEFAULT_TIME_COL, feature_col])
        .groupby(DEFAULT_TIME_COL)[feature_col]
        .mean()
        .reset_index()
        .sort_values(DEFAULT_TIME_COL)
    )

time_subplot_titles = [
    'Net Sentiment',
    'Positive Language',
    'Negative Language',
    'Uncertainty Language',
    'Litigious Language',
    'Constraining Language',
]
fig_time_subplots = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=time_subplot_titles,
    vertical_spacing=0.10,
)
fig_time_subplots.add_trace(
    go.Scatter(
        x=sentiment_over_time[DEFAULT_TIME_COL],
        y=sentiment_over_time[DEFAULT_SENTIMENT_COL],
        mode='lines+markers',
        name='sentiment_mean',
    ),
    row=1,
    col=1,
)
subplot_positions = {
    'positive_mean': (1, 2),
    'negative_mean': (2, 1),
    'uncertainty_mean': (2, 2),
    'litigious_mean': (3, 1),
    'constraining_mean': (3, 2),
}
for feature_col, (row_i, col_i) in subplot_positions.items():
    if feature_col not in time_series_by_feature:
        continue
    feature_df = time_series_by_feature[feature_col]
    fig_time_subplots.add_trace(
        go.Scatter(
            x=feature_df[DEFAULT_TIME_COL],
            y=feature_df[feature_col],
            mode='lines+markers',
            name=feature_col,
            showlegend=False,
        ),
        row=row_i,
        col=col_i,
    )
fig_time_subplots.update_layout(
    height=950,
    width=1000,
    title='Constitution Sentiment Dimensions by Year Created',
    template='plotly_white',
)
fig_time_subplots.update_xaxes(title_text='Year Created')
fig_time_subplots.update_yaxes(title_text='Mean Score', row=1, col=1)
fig_time_subplots.update_yaxes(title_text='Mean Score', row=1, col=2)
fig_time_subplots.update_yaxes(title_text='Mean Score', row=2, col=1)
fig_time_subplots.update_yaxes(title_text='Mean Score', row=2, col=2)
fig_time_subplots.update_yaxes(title_text='Mean Score', row=3, col=1)
fig_time_subplots.update_yaxes(title_text='Mean Score', row=3, col=2)
fig_time_subplots.show()

# Compare the distribution of constitution-level sentiment across broad regions.
region_sent = COUNTRY_SENT.dropna(subset=['region_compressed', DEFAULT_SENTIMENT_COL]).copy()
fig_region = px.box(
    region_sent,
    x='region_compressed',
    y=DEFAULT_SENTIMENT_COL,
    color='region_compressed',
    points='all',
    title='Constitution Sentiment Distribution by Region'
)
fig_region.update_layout(
    xaxis_title='Region',
    yaxis_title='Mean Sentiment',
    template='plotly_white',
    showlegend=False
)
fig_region.show()

# Compare the richer Loughran-McDonald dimensions across regime categories.
profile_cols = [
    col for col in [
        'positive_mean', 'negative_mean', 'uncertainty_mean', 'litigious_mean',
        'strong_modal_mean', 'weak_modal_mean', 'constraining_mean', 'complexity_mean'
    ] if col in COUNTRY_SENT.columns
]
regime_profiles = (
    COUNTRY_SENT.dropna(subset=['v2x_regime_cat'])
    .query("v2x_regime_cat != 'Unknown'")
    .groupby('v2x_regime_cat')[profile_cols]
    .mean()
)
fig_heatmap = px.imshow(
    regime_profiles,
    aspect='auto',
    color_continuous_scale='RdBu_r',
    origin='lower',
    title='Mean Loughran-McDonald Profile by V-Dem Regime Category'
)
fig_heatmap.update_layout(
    xaxis_title='Lexicon Dimension',
    yaxis_title='Regime Category',
    template='plotly_white'
)
fig_heatmap.show()

# Test whether constitutions with different rule-of-law scores also differ in net sentiment.
rule_scatter = COUNTRY_SENT.dropna(subset=['v2x_rule', DEFAULT_SENTIMENT_COL]).copy()
fig_rule = px.scatter(
    rule_scatter,
    x='v2x_rule',
    y=DEFAULT_SENTIMENT_COL,
    color='v2x_regime_cat',
    hover_name='country_id',
    title='Constitution Sentiment vs Rule of Law Score'
)
fig_rule.update_layout(
    xaxis_title='V-Dem Rule of Law',
    yaxis_title='Mean Sentiment',
    template='plotly_white'
)
fig_rule.show()

display(sentiment_over_time.tail())
display(regime_profiles.round(3))

,year_created,sentiment_mean
70,2010,-0.021085
71,2011,-0.015190
72,2012,-0.017845
73,2013,-0.028060
74,2014,-0.015531


,positive_mean,negative_mean,uncertainty_mean,litigious_mean,strong_modal_mean,weak_modal_mean,constraining_mean,complexity_mean
v2x_regime_cat,,,,,,,,
Closed autocracy,0.011,0.028,0.011,0.074,0.003,0.009,0.010,0.002
Electoral autocracy,0.009,0.033,0.013,0.083,0.003,0.011,0.010,0.003
Electoral democracy,0.008,0.031,0.014,0.080,0.002,0.013,0.011,0.003
Liberal democracy,0.006,0.034,0.018,0.079,0.003,0.016,0.010,0.002


## Save Outputs

In [11]:
# Save each sentiment table in parquet format for reuse in the final project notebook and riffs.
VOCAB_SENT.reset_index().to_parquet(VOCAB_SENT_PARQUET, index=False)
BOW_SENT.to_parquet(BOW_SENT_PARQUET, index=False)
DOC_SENT.to_parquet(DOC_SENT_PARQUET, index=False)

# Save the main year-created sentiment dimensions figure as interactive HTML for embedding.
fig_time_subplots.write_html('sentiment_dimensions_by_year_created.html', include_plotlyjs=True)

print(f'Saved VOCAB_SENT to: {VOCAB_SENT_PARQUET.resolve()}')
print(f'Saved BOW_SENT to: {BOW_SENT_PARQUET.resolve()}')
print(f'Saved DOC_SENT to: {DOC_SENT_PARQUET.resolve()}')
print(f'Saved sentiment figure to: {(Path("sentiment_dimensions_by_year_created.html")).resolve()}')

Saved VOCAB_SENT to: C:\Users\garre\school\spring_2026\ds_5001\vocab_sent.parquet
Saved BOW_SENT to: C:\Users\garre\school\spring_2026\ds_5001\bow_sent.parquet
Saved DOC_SENT to: C:\Users\garre\school\spring_2026\ds_5001\doc_sent.parquet
Saved sentiment figure to: C:\Users\garre\school\spring_2026\ds_5001\sentiment_dimensions_by_year_created.html
